# Chapter 3 — Chains

Hands-On LangChain. Code targets **LangChain 1.x** (LCEL — the `|` pipe).

Companion notebook for Chapter 3. Run top to bottom. Requires `OPENAI_API_KEY` and `MODEL` in `.env`,
and the dataset at `../data/product-reviews.csv`.

## 3.0 Setup

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv(find_dotenv(usecwd=True))
llm_model = os.getenv("MODEL", "gpt-5.4-mini")

In [2]:
df = pd.read_csv("../data/product-reviews.csv")
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld,I loved this product. But they only seem to la...


## 3.1 A basic chain

The most fundamental "chain" combines a **prompt** and a **model**. In LangChain 1.x you build it with
the pipe operator (`|`) — this is the LangChain Expression Language (LCEL). We add `StrOutputParser()`
so the chain returns a plain string instead of a message object.

We use a high temperature here to get more diverse, creative answers.

In [3]:
llm = ChatOpenAI(temperature=0.9, model=llm_model)

prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)

chain = prompt | llm | StrOutputParser()

In [4]:
product = "Deep Learning GPUs"
chain.invoke({"product": product})

'A good name should sound **technical, powerful, and AI-focused**. If you mean a **category/description** of such a company, the best term is usually:\n\n**AI hardware company**  \nor more specifically:  \n**GPU accelerator company** / **deep learning accelerator company**\n\nIf you mean a **brand name**, here are some strong naming directions:\n\n- **NexaCore**\n- **DeepForge**\n- **TensorWorks**\n- **NeuronGrid**\n- **AstraGPU**\n- **InferX**\n- **CoreMind**\n- **VoltTensor**\n- **QuantumLane**\n- **SynaptiCore**\n\nIf you want, I can also give you:\n1. **more name ideas**,  \n2. **names that sound like Nvidia/AMD style brands**, or  \n3. **a name optimized for startups vs. enterprise**.'

## 3.2 Sequential chains

Often you want to run several steps in order, feeding the output of one into the next.

### A simple two-step sequence

Chain one turns a product into a company name; chain two writes a short description of that company.
We connect them by mapping chain one's string output into the `{company_name}` slot that chain two
expects.

In [5]:
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)
second_prompt = ChatPromptTemplate.from_template(
    "Write a 30 word description for the following company: {company_name}"
)

# chain one produces the name; we feed it into chain two as `company_name`
overall_simple_chain = (
    {"company_name": first_prompt | llm | StrOutputParser()}
    | second_prompt
    | llm
    | StrOutputParser()
)

In [6]:
overall_simple_chain.invoke({"product": product})

'An AI accelerator company designs and manufactures specialized GPU hardware optimized for deep learning, enabling faster model training, efficient inference, and high-performance computing across modern AI workloads.'

### A multi-step sequence with several inputs and outputs

What if later steps need *several* earlier results, and you want to keep all the intermediate values?
The idiom for this is `RunnablePassthrough.assign`, which runs a sub-chain and **adds its result as a
new key** to the data flowing through — so every later step can read every earlier result by name.

We will take a (possibly non-English) product review and, in sequence:

1. translate it to English (`English_Review`),
2. summarize it in one sentence (`summary`),
3. detect its original language (`language`),
4. write a follow-up reply in that original language (`followup_message`).

In [7]:
from langchain_core.runnables import RunnablePassthrough

translate_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:\n\n{Review}"
)
summary_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:\n\n{English_Review}"
)
language_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
followup_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)

# each .assign adds one key; later prompts read keys produced earlier
overall_chain = (
    RunnablePassthrough.assign(English_Review=translate_prompt | llm | StrOutputParser())
    .assign(summary=summary_prompt | llm | StrOutputParser())
    .assign(language=language_prompt | llm | StrOutputParser())
    .assign(followup_message=followup_prompt | llm | StrOutputParser())
)

In [8]:
review = df.Review[5]
result = overall_chain.invoke({"Review": review})
result

{'Review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?",
 'English_Review': 'I find the taste mediocre. The foam doesn’t hold, which is odd. I buy the same ones in stores and the taste is much better...\nOld batch or counterfeit!?',
 'summary': 'The reviewer says the product tastes mediocre, the foam doesn’t hold properly, and they suspect it may be an old batch or counterfeit because store-bought versions taste better.',
 'language': 'The review is in **French**.',
 'followup_message': 'Merci pour votre retour. Nous sommes désolés d’apprendre que le produit n’a pas répondu à vos attentes, notamment en ce qui concerne le goût et la tenue de la mousse.\n\nCe que vous décrivez n’est pas conforme à l’expérience que nous souhaitons offrir. Il est possible qu’il s’agisse d’un lot ancien ou d’un produit non authentique, ce qui pourrait expliquer la différence avec les vers

## 3.3 Routing between chains

Sometimes you want to send the input to a *different* specialized chain depending on what it is — a
physics question to a physics expert, a math question to a mathematician, and so on. This is **routing**.

In LangChain 1.x the clean way to route is:

1. a small **classifier** chain that decides the topic (we force a fixed set of labels with
   `with_structured_output`), then
2. a plain Python function that picks the matching destination chain.

First, the four specialist prompts and their chains.

In [9]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{input}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. You have an excellent knowledge of and \
understanding of people, events and contexts from a range of historical periods.

Here is a question:
{input}"""

cs_template = """You are a successful computer scientist. You are great at answering coding \
questions because you can describe a solution in clear imperative steps and balance time and \
space complexity.

Here is a question:
{input}"""

llm0 = ChatOpenAI(temperature=0, model=llm_model)

destination_chains = {
    "physics": ChatPromptTemplate.from_template(physics_template) | llm0 | StrOutputParser(),
    "math": ChatPromptTemplate.from_template(math_template) | llm0 | StrOutputParser(),
    "history": ChatPromptTemplate.from_template(history_template) | llm0 | StrOutputParser(),
    "computer science": ChatPromptTemplate.from_template(cs_template) | llm0 | StrOutputParser(),
}
# default chain for anything that doesn't fit a specialist
default_chain = ChatPromptTemplate.from_template("{input}") | llm0 | StrOutputParser()

Now the classifier. We use a Pydantic schema with a constrained `Literal` field, so the model can
only return one of our known destinations (or `DEFAULT`).

In [10]:
from typing import Literal
from pydantic import BaseModel, Field

class RouteQuery(BaseModel):
    """Pick the best destination for the user's question."""
    destination: Literal["physics", "math", "history", "computer science", "DEFAULT"] = Field(
        description="The subject best suited to answer the question; DEFAULT if none fit."
    )

router_prompt = ChatPromptTemplate.from_template(
    "Given the user question below, classify it as one of: physics, math, history, "
    "computer science, or DEFAULT.\n\nQuestion: {input}"
)
router = router_prompt | llm0.with_structured_output(RouteQuery)

Finally, a routing function ties it together: classify, then invoke the chosen chain.

In [11]:
from langchain_core.runnables import RunnableLambda

def route(info):
    destination = router.invoke({"input": info["input"]}).destination
    chain = destination_chains.get(destination, default_chain)
    print(f"[router] -> {destination}")
    return chain.invoke({"input": info["input"]})

full_chain = RunnableLambda(route)

In [12]:
print(full_chain.invoke({"input": "What is the speed of light?"}))

[router] -> physics


The speed of light in vacuum is exactly:

**299,792,458 meters per second**

That’s about **3.00 × 10^8 m/s**, or **about 300,000 kilometers per second**.

If you want, I can also explain why this number is so important in physics.


In [13]:
print(full_chain.invoke({"input": "What is 2 + 2 times 8?"}))

[router] -> math


Using order of operations, do multiplication before addition:

- \(2 \times 8 = 16\)
- \(2 + 16 = 18\)

So the answer is **18**.
